# Resident Risk Model — Inference

Loads `risk_model.sav` and scores all residents, writing predictions to `operational.resident_risk_predictions`.

| Column | Description |
|---|---|
| `predicted_risk` | Low / Medium / High / Critical |
| `prob_low` – `prob_critical` | Per-class probabilities |

In [ ]:
import sys
sys.path.insert(0, '../jobs')

import json
from datetime import datetime, timezone

import joblib
import pandas as pd

from config import OPERATIONAL_SCHEMA, RISK_METADATA_PATH, RISK_MODEL_PATH
from utils_db import ensure_risk_predictions_table, get_engine, pg_conn

print('Imports loaded.')

## Load model

In [ ]:
model = joblib.load(str(RISK_MODEL_PATH))
with open(RISK_METADATA_PATH) as f:
    metadata = json.load(f)

feature_cols = metadata['feature_cols']
classes      = metadata['classes']
print(f"Model v{metadata['model_version']} | classes: {classes}")

## Build features from operational schema

In [ ]:
from inference import _build_features

engine = get_engine(OPERATIONAL_SCHEMA)
df = _build_features(engine)
print(f'Scoring {len(df)} residents')
df[feature_cols].head()

## Run predictions

In [ ]:
X      = df[feature_cols]
probs  = model.predict_proba(X)
preds  = model.predict(X)
ts     = datetime.now(timezone.utc).isoformat()

class_to_col = {'Low': 'prob_low', 'Medium': 'prob_medium', 'High': 'prob_high', 'Critical': 'prob_critical'}
prob_df = pd.DataFrame(probs, columns=classes)

rows = []
for i, row in df.iterrows():
    idx = list(df.index).index(i)
    row_probs = {col: float(prob_df.loc[i, cls]) for cls, col in class_to_col.items() if cls in prob_df.columns}
    rows.append((
        int(row['resident_id']), str(preds[idx]),
        row_probs.get('prob_low'), row_probs.get('prob_medium'),
        row_probs.get('prob_high'), row_probs.get('prob_critical'), ts,
    ))

print(f'Built {len(rows)} prediction rows')

## Write to database

In [ ]:
ensure_risk_predictions_table(OPERATIONAL_SCHEMA)

with pg_conn(OPERATIONAL_SCHEMA) as conn:
    with conn.cursor() as cur:
        cur.executemany("""
            INSERT INTO operational.resident_risk_predictions
                (resident_id, predicted_risk, prob_low, prob_medium, prob_high, prob_critical, prediction_ts)
            VALUES (%s, %s, %s, %s, %s, %s, %s)
            ON CONFLICT (resident_id) DO UPDATE SET
                predicted_risk = EXCLUDED.predicted_risk,
                prob_low       = EXCLUDED.prob_low,
                prob_medium    = EXCLUDED.prob_medium,
                prob_high      = EXCLUDED.prob_high,
                prob_critical  = EXCLUDED.prob_critical,
                prediction_ts  = EXCLUDED.prediction_ts
        """, rows)

print(f'Wrote {len(rows)} predictions to resident_risk_predictions')
print(f'Timestamp: {ts}')